# 55 Optode co-registration invariance

For each optode-cap subject we run the anonymization pipeline once via `run_pipeline` and pull two head-isolated meshes from the artifact bundle: `surface_ctf` (head-isolated, CTF frame, **before** face-mask deletion) and `surface_anon_ctf` (same head-isolated mesh, **after** face-mask deletion). `ColoredStickerProcessor` is then run on both, sticker centres are matched between the two runs by nearest neighbour, and per-optode Euclidean deviations are reported for both the raw sticker centres and the scalp-projected optode positions (sticker centre minus optode length along the surface normal).

Both inputs come from the same in-memory pipeline run and share the same head-isolation step, the same CTF alignment, and the same un-recompressed original texture; the only geometric difference between them is the set of vertices removed by `delete_masked_vertices`. This matters: comparing the raw Einstar OBJ (which includes neck, shoulders, and clothing) against the head-only anonymized output would let `ColoredStickerProcessor` pick up yellow false positives on the body, inflating the sticker count on the original side and producing nearest-neighbour matching artifacts when the unmatched phantoms get paired with real cap stickers on the opposite side of the head. Sharing head-isolation removes that confound.

**Cohort restriction.** This test detects yellow sticker markers mounted on the optode cap. Only the optode-cap cohort (S1--S7, Subject 16--22) carries those markers; the bare-cap cohort (S8--S11, Subject 12--15) wears a cap without optodes mounted, so the test is undefined for them and they are intentionally skipped here. The other evaluation notebooks (51, 52, 53, 54, 56, 59) iterate over the full eleven-subject cohort.

Populates Table 4.5 of the thesis (Section 4.4.3). Expected outcome: zero deviation up to numerical noise, because every sticker sits on the cap which lies above the cap-boundary plane and therefore outside the facial mask by construction.

Output: `thesis_results_out/coreg_invariance.csv`, with one row per optode-cap subject and an `optode=True` column for schema parity with the other Results-chapter CSVs.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    OPTODE_SUBJECTS, subject_paths, load_raw, load_landmarks, run_pipeline,
    is_optode, s_id,
)

import numpy as np
import pandas as pd

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

import cedalion
from cedalion.geometry.photogrammetry.processors import (
    ColoredStickerProcessor,
)
from scipy.spatial import KDTree

# Yellow optode stickers, same HSV range as notebook 41.
COLORS = {'O': (0.11, 0.21, 0.7, 1.0)}
OPTODE_LENGTH = 22.6 * cedalion.units.mm

## 1. Sticker detection wrapper

Returns sticker centres and normals as plain numpy arrays for easy nearest-neighbour matching.

In [2]:
def detect_stickers(surface):

    proc = ColoredStickerProcessor(colors=COLORS)

    centres, normals = proc.process(surface, details=False)

    c_np = centres.pint.dequantify().values

    n_np = normals.values if hasattr(normals, 'values') else np.asarray(normals)

    return c_np, n_np



def match_by_nn(a, b):

    '''For each row of a, return the index of the nearest row of b.'''

    tree = KDTree(b)

    _, idx = tree.query(a)

    return idx

## 2. Per-subject comparison

In [3]:
rows = []
for n in OPTODE_SUBJECTS:
    paths = subject_paths(n)
    if not paths.raw_exists:
        print(f'skipping Subject{n}: missing raw scan')
        continue
    if not paths.landmarks_exist:
        print(f'skipping Subject{n}: missing landmarks TSV')
        continue
    print(f'--- {s_id(n)} (Subject{n}) ---')

    art = run_pipeline(load_raw(n), load_landmarks(n), subject=n)
    surface_orig = art.surface_ctf       # head-isolated, CTF, before face deletion
    surface_anon = art.surface_anon_ctf  # head-isolated, CTF, after face deletion

    c_orig, n_orig = detect_stickers(surface_orig)
    c_anon, n_anon = detect_stickers(surface_anon)

    if len(c_orig) == 0 or len(c_anon) == 0:
        print(f'  no stickers detected on one side '
              f'(orig={len(c_orig)}, anon={len(c_anon)})')
        continue

    idx = match_by_nn(c_orig, c_anon)
    sticker_dev = np.linalg.norm(c_orig - c_anon[idx], axis=1)

    L = float(OPTODE_LENGTH.to('mm').magnitude)
    scalp_orig = c_orig - L * n_orig
    scalp_anon = c_anon[idx] - L * n_anon[idx]
    scalp_dev = np.linalg.norm(scalp_orig - scalp_anon, axis=1)

    rows.append({
        's_id': s_id(n),
        'subject': n,
        'optode': is_optode(n),
        'n_stickers_original': len(c_orig),
        'n_stickers_anonymized': len(c_anon),
        'n_matched': len(idx),
        'sticker_mean_mm': float(sticker_dev.mean()),
        'sticker_median_mm': float(np.median(sticker_dev)),
        'sticker_max_mm': float(sticker_dev.max()),
        'scalp_mean_mm': float(scalp_dev.mean()),
        'scalp_median_mm': float(np.median(scalp_dev)),
        'scalp_max_mm': float(scalp_dev.max()),
    })
    print(rows[-1])

--- S1 (Subject16) ---


[[ -18.00455171 -101.35689301  -49.90083092]
 [ -17.36425037 -101.26923325  -50.40896743]
 [ -17.58280801 -100.98613082  -51.14345664]
 ...
 [-118.10573076   39.40446198   33.01622935]
 [-121.2215538    33.19909023   55.6572317 ]
 [ -32.02600462   65.39815272  -46.98181825]]
[[-83.62772362 -39.18978661 122.40603235]
 [ 17.59523849 -41.4593695  154.15166116]
 [-75.93538423 -40.2388502  128.16637636]
 ...
 [  3.40884444 -80.66966494 132.99746232]
 [  2.96306468 -80.70235116 133.05590369]
 [  2.20979903 -80.741302   132.99769018]]
O (0.11, 0.21, 0.7, 1.0)
[0.80864561 0.25652499 0.90779208 0.99456393 0.88376101 0.63718142
 0.32089567 0.62904539 0.18573855 1.15660754 0.1536981  0.00922152
 0.56982297 0.64168943 0.46328306 0.61566801 0.87477079 0.15965823
 0.11353025]
3.574345116866262
surface.crs ctf


/home/ma7/.conda/envs/cedalion/lib/python3.11/site-packages/xarray/core/variable.py:315: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


[[ -18.00455171 -101.35689301  -49.90083092]
 [ -17.36425037 -101.26923325  -50.40896743]
 [ -17.58280801 -100.98613082  -51.14345664]
 ...
 [-118.10573076   39.40446198   33.01622935]
 [-121.2215538    33.19909023   55.6572317 ]
 [ -32.02600462   65.39815272  -46.98181825]]
[[-83.62772362 -39.18978661 122.40603235]
 [ 17.59523849 -41.4593695  154.15166116]
 [-75.93538423 -40.2388502  128.16637636]
 ...
 [  3.40884444 -80.66966494 132.99746232]
 [  2.96306468 -80.70235116 133.05590369]
 [  2.20979903 -80.741302   132.99769018]]
O (0.11, 0.21, 0.7, 1.0)
[0.80864561 0.25652499 0.90779208 0.99456393 0.88376101 0.63718142
 0.32089567 0.62904539 0.18573855 1.15660754 0.1536981  0.00922152
 0.56982297 0.64168943 0.46328306 0.61566801 0.87477079 0.15965823
 0.11353025]
3.574345116866262
surface.crs ctf
{'s_id': 'S1', 'subject': 16, 'optode': True, 'n_stickers_original': 19, 'n_stickers_anonymized': 19, 'n_matched': 19, 'sticker_mean_mm': 0.0, 'sticker_median_mm': 0.0, 'sticker_max_mm': 0.0, '

/home/ma7/.conda/envs/cedalion/lib/python3.11/site-packages/xarray/core/variable.py:315: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


...

/home/ma7/.conda/envs/cedalion/lib/python3.11/site-packages/xarray/core/variable.py:315: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


[[-105.74027664 -103.51566261  -61.31412435]
 [ -29.30742122   99.47352464   70.94195514]
 [ -92.42476339  -10.48506195  125.02474163]
 ...
 [  13.18290011   14.93265513  133.13134391]
 [-100.37320932   10.39548131  110.67392679]
 [-120.62717017  -56.90471638  -14.17351366]]
[[ 50.88536149 -33.05322285 132.2957552 ]
 [  6.88961332 100.94044927  74.83482891]
 [  6.98656925 100.8354989   75.20945163]
 ...
 [114.29429785  38.9490919   64.82357339]
 [114.47327892  39.0640598   64.3823771 ]
 [114.28826933  39.31171899  63.21836039]]
O (0.11, 0.21, 0.7, 1.0)
[0.12055334 1.16284857 1.48791388 0.51091437 1.39303198 0.23838312
 0.4098559  0.28923456 0.3477276  1.51396856 1.61867151 0.49111139
 0.20229273 0.39042139 0.82300073 1.38996161 0.44898889 0.33806708
 0.17742365 0.22163885 0.2498382  0.53088983 0.27922052 0.13616846
 0.3668017  0.82408899 0.33818201 0.26384039 0.36703333 0.89757593
 1.02417181]
4.128625242171749
surface.crs ctf


/home/ma7/.conda/envs/cedalion/lib/python3.11/site-packages/xarray/core/variable.py:315: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


[[-105.74027664 -103.51566261  -61.31412435]
 [ -29.30742122   99.47352464   70.94195514]
 [ -92.42476339  -10.48506195  125.02474163]
 ...
 [  13.18290011   14.93265513  133.13134391]
 [-100.37320932   10.39548131  110.67392679]
 [-120.62717017  -56.90471638  -14.17351366]]
[[ 50.88536149 -33.05322285 132.2957552 ]
 [  6.88961332 100.94044927  74.83482891]
 [  6.98656925 100.8354989   75.20945163]
 ...
 [114.29429785  38.9490919   64.82357339]
 [114.47327892  39.0640598   64.3823771 ]
 [114.28826933  39.31171899  63.21836039]]
O (0.11, 0.21, 0.7, 1.0)
[0.12055334 1.16284857 1.48791388 0.51091437 1.39303198 0.23838312
 0.4098559  0.28923456 0.3477276  1.51396856 1.61867151 0.49111139
 0.20229273 0.39042139 0.82300073 1.38996161 0.44898889 0.33806708
 0.17742365 0.22163885 0.2498382  0.53088983 0.27922052 0.13616846
 0.3668017  0.82408899 0.33818201 0.26384039 0.36703333 0.89757593
 1.02417181]
4.128625242171749
surface.crs ctf
{'s_id': 'S7', 'subject': 22, 'optode': True, 'n_stickers_o

/home/ma7/.conda/envs/cedalion/lib/python3.11/site-packages/xarray/core/variable.py:315: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


## 3. Summary

In [4]:
df = pd.DataFrame(rows)
# Order rows by S-ID (S1..S7) -- helper-controlled, matches thesis tables.
df = df.iloc[df['s_id'].map(lambda s: int(s[1:])).argsort()].reset_index(drop=True)
df

,s_id,subject,optode,n_stickers_original,n_stickers_anonymized,n_matched,sticker_mean_mm,sticker_median_mm,sticker_max_mm,scalp_mean_mm,scalp_median_mm,scalp_max_mm
0,S1,16,True,19,19,19,0.0,0.0,0.0,0.0,0.0,0.0
1,S2,17,True,21,21,21,0.0,0.0,0.0,0.0,0.0,0.0
2,S3,18,True,15,15,15,0.0,0.0,0.0,0.0,0.0,0.0
3,S4,19,True,12,12,12,0.0,0.0,0.0,0.0,0.0,0.0
4,S5,20,True,14,14,14,0.0,0.0,0.0,0.0,0.0,0.0
5,S6,21,True,19,19,19,0.0,0.0,0.0,0.0,0.0,0.0
6,S7,22,True,31,31,31,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Save

In [5]:
out = OUT_DIR / 'coreg_invariance.csv'

df.to_csv(out, index=False)

print(f'Wrote {out}')

Wrote thesis_results_out/coreg_invariance.csv
